In [1]:
import numpy as np
import polars as pl

In [2]:
pl.__version__

'1.8.2'

HF_ENDPOINT="http://huggingface.proxy" hf download deepvk/VK-LSVD --repo-type dataset --include "metadata/*" --local-dir  /home/jovyan/IRec/sigir/lsvd_data/raw

HF_ENDPOINT="http://huggingface.proxy" hf download deepvk/VK-LSVD --repo-type dataset --include "subsamples/ur0.01_ir0.01/*" --local-dir  /home/jovyan/IRec/sigir/lsvd_data/raw


Разбиение сабсэмплов на базовую, гэп, вал и тест части

Добавляется колонка original_order чтобы сохранять порядок внутри каждой из частей

In [ ]:
subsample_name = 'ur0.01_ir0.01'
content_embedding_size = 256
DATASET_PATH = "/home/jovyan/IRec/sigir/lsvd_data/raw"

metadata_files = ['metadata/users_metadata.parquet',
                  'metadata/items_metadata.parquet',
                  'metadata/item_embeddings.npz']

BASE_WEEKS = (15, 23)
GAP_WEEKS = (23, 24) #увеличить гэп
VAL_WEEKS = (24, 25)

base_interactions_files = [f'subsamples/{subsample_name}/train/week_{i:02}.parquet'
                            for i in range(BASE_WEEKS[0], BASE_WEEKS[1])]

gap_interactions_files = [f'subsamples/{subsample_name}/train/week_{i:02}.parquet'
                            for i in range(GAP_WEEKS[0], GAP_WEEKS[1])]

val_interactions_files = [f'subsamples/{subsample_name}/train/week_{i:02}.parquet'
                            for i in range(VAL_WEEKS[0], VAL_WEEKS[1])]

test_interactions_files = [f'subsamples/{subsample_name}/validation/week_25.parquet']

all_interactions_files = base_interactions_files + gap_interactions_files + val_interactions_files + test_interactions_files

base_with_gap_interactions_files = base_interactions_files + gap_interactions_files

len(base_interactions_files), len(gap_interactions_files), len(val_interactions_files), len(test_interactions_files), len(all_interactions_files), len(base_with_gap_interactions_files)

(8, 1, 1, 1, 11, 9)

In [8]:
def get_parquet_interactions(data_files):
    data_interactions = pl.concat([pl.scan_parquet(f'{DATASET_PATH}/{file}')
                                for file in data_files])
    data_interactions = data_interactions.collect(streaming=True)
    data_interactions = data_interactions.with_row_index("original_order")
    return data_interactions


In [9]:
base_interactions = get_parquet_interactions(base_interactions_files)
gap_interactions = get_parquet_interactions(gap_interactions_files)
val_interactions = get_parquet_interactions(val_interactions_files)
test_interactions = get_parquet_interactions(test_interactions_files)
all_data_interactions = get_parquet_interactions(all_interactions_files)
base_with_gap_interactions = get_parquet_interactions(base_with_gap_interactions_files)

Загрузка и фильтрация эмбеддингов

In [10]:
all_data_users = all_data_interactions.select('user_id').unique()
all_data_items = all_data_interactions.select('item_id').unique()

item_ids = np.load(f"{DATASET_PATH}/metadata/item_embeddings.npz")['item_id']
item_embeddings = np.load(f"{DATASET_PATH}/metadata/item_embeddings.npz")['embedding']

mask = np.isin(item_ids, all_data_items.to_numpy())
item_ids = item_ids[mask]
item_embeddings = item_embeddings[mask]
item_embeddings = item_embeddings[:, :content_embedding_size]

users_metadata = pl.read_parquet(f"{DATASET_PATH}/metadata/users_metadata.parquet")
items_metadata = pl.read_parquet(f"{DATASET_PATH}/metadata/items_metadata.parquet")

users_metadata = users_metadata.join(all_data_users, on='user_id')
items_metadata = items_metadata.join(all_data_items, on='item_id')
items_metadata = items_metadata.join(pl.DataFrame({'item_id': item_ids, 
                                                   'embedding': item_embeddings}), on='item_id')

Сжатие айтем айди и ремапинг

In [11]:
all_data_items = all_data_interactions.select('item_id').unique()
all_data_users = all_data_interactions.select('user_id').unique()

unique_items_sorted = all_data_items.sort('item_id').with_row_index('new_item_id')
global_item_mapping = dict(zip(unique_items_sorted['item_id'], unique_items_sorted['new_item_id']))

print(f"Total users: {all_data_users.shape[0]}, Total items: {len(global_item_mapping)}")

Total users: 79074, Total items: 62758


In [12]:
def remap_interactions(df, mapping):
    return df.with_columns(
        pl.col('item_id')
        .map_elements(lambda x: mapping.get(x, None), return_dtype=pl.UInt32)
    )

base_interactions_remapped = remap_interactions(base_interactions, global_item_mapping)
gap_interactions_remapped = remap_interactions(gap_interactions, global_item_mapping)
test_interactions_remapped = remap_interactions(test_interactions, global_item_mapping)
val_interactions_remapped = remap_interactions(val_interactions, global_item_mapping)
all_data_interactions_remapped = remap_interactions(all_data_interactions, global_item_mapping)

In [13]:
del base_interactions, gap_interactions, test_interactions, val_interactions, all_data_interactions

In [14]:
base_with_gap_interactions_remapped = remap_interactions(base_with_gap_interactions, global_item_mapping)
del base_with_gap_interactions

In [15]:
items_metadata_remapped = remap_interactions(items_metadata, global_item_mapping)

Группировка по юзер айди

In [16]:
def get_grouped_interactions(data_interactions):
    print(f"interactions count: {data_interactions.shape}")
    data_res = (
        data_interactions
        .select(['original_order', 'user_id', 'item_id'])
        .group_by('user_id')
        .agg(
            pl.col('item_id')
            .sort_by(pl.col('original_order'))
            .alias('item_ids'),
            pl.col('original_order').alias('timestamps')
        )
        .rename({'user_id': 'uid'})
    )
    print(f"users count: {data_res.shape}")
    return data_res
base_interactions_grouped = get_grouped_interactions(base_interactions_remapped)
gap_interactions_grouped = get_grouped_interactions(gap_interactions_remapped)
test_interactions_grouped = get_grouped_interactions(test_interactions_remapped)
val_interactions_grouped = get_grouped_interactions(val_interactions_remapped)
all_data_interactions_grouped = get_grouped_interactions(all_data_interactions_remapped)

interactions count: (1244323, 13)
users count: (74862, 3)
interactions count: (176791, 13)
users count: (44444, 3)
interactions count: (170111, 13)
users count: (43370, 3)
interactions count: (164151, 13)
users count: (43233, 3)
interactions count: (1755376, 13)
users count: (79074, 3)


In [17]:
base_interactions_grouped.head(1)

uid,item_ids,timestamps
u32,list[u32],list[u32]
2655558,"[16621, 42990, … 51285]","[46109, 59132, … 1209536]"


In [18]:
del base_interactions_remapped, gap_interactions_remapped, test_interactions_remapped, val_interactions_remapped

In [19]:
base_with_gap_interactions_grouped = get_grouped_interactions(base_with_gap_interactions_remapped)
del base_with_gap_interactions_remapped

interactions count: (1421114, 13)
users count: (76483, 3)


Сохранение

In [20]:
import json
OUTPUT_DIR = "/home/jovyan/IRec/sigir/lsvd_data/8-days-base-ows"

mapping_output_path = f"{OUTPUT_DIR}/global_item_mapping.json"

with open(mapping_output_path, 'w') as f:
    json.dump({str(k): v for k, v in global_item_mapping.items()}, f, indent=2)

print(f"Сохранён маппинг: {mapping_output_path}")

Сохранён маппинг: /home/jovyan/IRec/sigir/lsvd_data/8-days-base-ows/global_item_mapping.json


In [21]:
def write_parquet(output_dir, data, file_name):
    output_parquet_path = f"{output_dir}/{file_name}.parquet"
    data.write_parquet(output_parquet_path)
    print(f"Сохранен файл: {file_name}")

write_parquet(OUTPUT_DIR, items_metadata_remapped, "items_metadata_remapped")
write_parquet(OUTPUT_DIR, items_metadata, "items_metadata_remapped_old")

write_parquet(OUTPUT_DIR, base_interactions_grouped, "base_interactions_grouped")
write_parquet(OUTPUT_DIR, gap_interactions_grouped, "gap_interactions_grouped")
write_parquet(OUTPUT_DIR, test_interactions_grouped, "test_interactions_grouped")
write_parquet(OUTPUT_DIR, val_interactions_grouped, "val_interactions_grouped")
write_parquet(OUTPUT_DIR, base_with_gap_interactions_grouped, "base_with_gap_interactions_grouped")

write_parquet(OUTPUT_DIR, all_data_interactions_grouped, "all_data_interactions_grouped")

write_parquet(OUTPUT_DIR, all_data_interactions_remapped, "all_data_interactions_remapped")

Сохранен файл: items_metadata_remapped
Сохранен файл: items_metadata_remapped_old
Сохранен файл: base_interactions_grouped
Сохранен файл: gap_interactions_grouped
Сохранен файл: test_interactions_grouped
Сохранен файл: val_interactions_grouped
Сохранен файл: base_with_gap_interactions_grouped
Сохранен файл: all_data_interactions_grouped
Сохранен файл: all_data_interactions_remapped


In [22]:
len(list(items_metadata_remapped.head(1)['embedding'].item())), len(list(items_metadata.head(1)['embedding'].item()))

(64, 64)

In [23]:
items_metadata_remapped.shape

(62758, 5)

In [24]:
items_metadata_remapped.head(1)

item_id,author_id,duration,train_interactions_rank,embedding
u32,u32,u8,u32,"array[f32, 64]"
0,1249424,9,771612,"[-0.503418, 0.201538, … 0.007988]"


In [25]:
base_with_gap_interactions_grouped.head()

uid,item_ids,timestamps
u32,list[u32],list[u32]
4465123,"[28298, 3829, … 28995]","[257260, 272293, … 1390041]"
3043171,"[8638, 23487, … 15086]","[6628, 11364, … 1370935]"
2757146,"[56345, 56828, … 37056]","[194522, 217739, … 1390752]"
1148408,"[40326, 42152]","[427153, 1367211]"
2537065,"[27766, 39966, … 19887]","[9428, 35459, … 1214991]"
